# Klasifikasi Gambar Sampah (Organic, Plastic, Paper)

Notebook ini melatih model **MobileNetV2 (transfer learning)** untuk mengenali 3 kategori sampah:
**Organic Waste**, **Plastic Waste**, dan **Paper Waste**.

Alur: unduh dataset → buat subset 3 kelas dan bagi train/val/test → latih model → evaluasi di data test
(accuracy, confusion matrix, classification report) → simpan model dan hasil evaluasi.

**Cara pakai di Google Colab:** Runtime → Change runtime type → pilih **T4 GPU**, lalu jalankan semua sel dari atas.

In [ ]:
import os, json, random, shutil, glob
from pathlib import Path
from datetime import datetime

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)   # CATAT versi ini, dipakai di ml/requirements.txt
print("GPU:", tf.config.list_physical_devices("GPU"))

## 1. Dataset

Sumber: **Garbage Classification (12 classes)** oleh Mostafa Mohamed di Kaggle  
`https://www.kaggle.com/datasets/mostafaabla/garbage-classification`

Dipakai 3 dari 12 kelas: `biological` (menjadi Organic Waste), `plastic`, dan `paper`.

**Cara A (disarankan): token Kaggle.**
1. Login ke kaggle.com, klik foto profil → **Settings** → bagian **API** (API Tokens) → **Generate New Token**.
2. Salin token yang muncul (hanya tampil sekali).
3. Jalankan sel di bawah, lalu tempel token saat diminta.

Opsional: simpan token sebagai **Colab Secret** bernama `KAGGLE_API_TOKEN` (ikon kunci di sisi kiri Colab, aktifkan *Notebook access*) supaya tidak perlu menempel lagi.

**Cara B:** unduh zip dataset secara manual dari tautan di atas, unggah ke `/content/raw/` lewat panel file Colab, lalu lewati sel unduh.

Jangan menulis token langsung di dalam sel dan jangan meng-commit-nya ke GitHub.

In [ ]:
# Cara A: unduh lewat Kaggle API dengan token baru
import os, getpass
!pip install -q -U kaggle

try:
    from google.colab import userdata
    os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")   # dari Colab Secret
except Exception:
    os.environ["KAGGLE_API_TOKEN"] = getpass.getpass("Tempel Kaggle API token: ")

!kaggle datasets download -d mostafaabla/garbage-classification -p /content/raw
!unzip -q -o /content/raw/garbage-classification.zip -d /content/raw

In [ ]:
# Folder asli -> (nama folder baru, nama kategori yang ditampilkan di aplikasi)
CLASS_MAP = {
    "biological": ("organic", "Organic Waste"),
    "plastic":    ("plastic", "Plastic Waste"),
    "paper":      ("paper",   "Paper Waste"),
}
PER_CLASS = 300                       # gambar per kelas (total 900, syarat minimal 150)
SPLIT = {"train": 0.70, "val": 0.15}  # sisanya (15%) untuk test

def find_dir(name):
    hits = [p for p in glob.glob(f"/content/raw/**/{name}", recursive=True) if os.path.isdir(p)]
    assert hits, f"Folder '{name}' tidak ditemukan di /content/raw"
    return hits[0]

OUT = Path("/content/dataset")
if OUT.exists():
    shutil.rmtree(OUT)

counts = {}
for src_name, (dst_name, _) in CLASS_MAP.items():
    items = sorted(
        f for f in glob.glob(os.path.join(find_dir(src_name), "*"))
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    )
    random.shuffle(items)
    items = items[:PER_CLASS]
    n = len(items)
    n_train, n_val = int(n * SPLIT["train"]), int(n * SPLIT["val"])
    parts = {
        "train": items[:n_train],
        "val":   items[n_train:n_train + n_val],
        "test":  items[n_train + n_val:],
    }
    for split, files_ in parts.items():
        d = OUT / split / dst_name
        d.mkdir(parents=True, exist_ok=True)
        for f in files_:
            shutil.copy(f, d / Path(f).name)
        counts[(split, dst_name)] = len(files_)

print(f"{'kelas':<10}{'train':>8}{'val':>8}{'test':>8}")
for _, (dst, _) in CLASS_MAP.items():
    print(f"{dst:<10}{counts[('train', dst)]:>8}{counts[('val', dst)]:>8}{counts[('test', dst)]:>8}")
print("Total gambar:", sum(counts.values()))

In [ ]:
# Contoh gambar tiap kelas (untuk dokumentasi)
RESULTS = Path("/content/artifacts/results")
MODELS = Path("/content/artifacts/models")
RESULTS.mkdir(parents=True, exist_ok=True)
MODELS.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for r, (_, (dst, label)) in enumerate(CLASS_MAP.items()):
    paths = sorted((OUT / "train" / dst).glob("*"))[:4]
    for c, p in enumerate(paths):
        axes[r, c].imshow(plt.imread(p))
        axes[r, c].axis("off")
        if c == 0:
            axes[r, c].set_title(label, loc="left", fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS / "dataset_samples.png", dpi=120)
plt.show()

## 2. Memuat data

Urutan kelas dari `image_dataset_from_directory` adalah **alfabet nama folder** (`organic`, `paper`, `plastic`).
Urutan inilah yang disimpan ke `class_names.json` dan dipakai API, jadi jangan diubah.
Data test memakai `shuffle=False` supaya urutan label cocok saat membuat confusion matrix.

In [ ]:
IMG_SIZE = (224, 224)
BATCH = 32

train_ds = keras.utils.image_dataset_from_directory(
    str(OUT / "train"), image_size=IMG_SIZE, batch_size=BATCH, shuffle=True, seed=SEED)
val_ds = keras.utils.image_dataset_from_directory(
    str(OUT / "val"), image_size=IMG_SIZE, batch_size=BATCH, shuffle=False)
test_ds = keras.utils.image_dataset_from_directory(
    str(OUT / "test"), image_size=IMG_SIZE, batch_size=BATCH, shuffle=False)

folder_names = train_ds.class_names
DISPLAY = {dst: label for dst, label in CLASS_MAP.values()}
class_labels = [DISPLAY[n] for n in folder_names]
print(folder_names, "->", class_labels)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)
test_ds = test_ds.prefetch(AUTOTUNE)

## 3. Model (transfer learning)

- Dasar: **MobileNetV2** berbobot ImageNet.
- Lapisan akhir diganti: GlobalAveragePooling → Dropout → Dense(3, softmax).
- Normalisasi (`Rescaling`) dan augmentasi ada **di dalam model**, jadi API cukup mengirim gambar 224×224 apa adanya (nilai 0 sampai 255).
- Tahap 1: latih lapisan akhir saja. Tahap 2: *fine-tuning* 30 lapisan terakhir dengan learning rate kecil.

In [ ]:
def build_model(num_classes):
    base = keras.applications.MobileNetV2(
        input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet")
    base.trainable = False

    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = layers.RandomFlip("horizontal")(inputs)
    x = layers.RandomRotation(0.1)(x)
    x = layers.RandomZoom(0.1)(x)
    x = layers.RandomContrast(0.1)(x)
    x = layers.Rescaling(1.0 / 127.5, offset=-1)(x)   # skala -1..1 untuk MobileNetV2
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs), base

model, base = build_model(len(class_labels))
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_accuracy", patience=3, restore_best_weights=True)

print("== Tahap 1: latih lapisan akhir ==")
h1 = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=[early_stop])

In [ ]:
print("== Tahap 2: fine-tuning ==")
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=8, callbacks=[early_stop])

In [ ]:
# Kurva training
acc = h1.history["accuracy"] + h2.history["accuracy"]
val_acc = h1.history["val_accuracy"] + h2.history["val_accuracy"]
loss = h1.history["loss"] + h2.history["loss"]
val_loss = h1.history["val_loss"] + h2.history["val_loss"]
split_at = len(h1.history["accuracy"]) - 0.5

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(acc, label="train"); ax[0].plot(val_acc, label="val")
ax[0].axvline(split_at, color="gray", linestyle="--", label="mulai fine-tuning")
ax[0].set_title("Accuracy"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(loss, label="train"); ax[1].plot(val_loss, label="val")
ax[1].axvline(split_at, color="gray", linestyle="--")
ax[1].set_title("Loss"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout()
plt.savefig(RESULTS / "training_curves.png", dpi=120)
plt.show()

## 4. Evaluasi pada data test

Data test **tidak pernah dipakai saat training**, sehingga angka di bawah ini adalah bukti akurasi model.
Target: accuracy minimal **80%**.

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_ds])
probs = model.predict(test_ds)
y_pred = probs.argmax(axis=1)

test_acc = float(accuracy_score(y_true, y_pred))
print(f"Test accuracy: {test_acc:.4f} ({test_acc * 100:.2f}%)")
print("Target >= 80%:", "TERPENUHI" if test_acc >= 0.80 else "BELUM TERPENUHI")

report_text = classification_report(y_true, y_pred, target_names=class_labels, digits=4)
print()
print(report_text)
(RESULTS / "classification_report.txt").write_text(
    f"Test accuracy: {test_acc:.4f}\n\n{report_text}", encoding="utf-8")

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel("Prediksi"); plt.ylabel("Label sebenarnya"); plt.title("Confusion Matrix (data test)")
plt.tight_layout()
plt.savefig(RESULTS / "confusion_matrix.png", dpi=120)
plt.show()

## 5. Simpan model dan hasil

File yang dihasilkan (dikemas dalam satu zip untuk diunduh):

- `models/waste_classifier.keras`: model terlatih
- `models/class_names.json`: urutan kelas (dibaca oleh API)
- `results/`: confusion matrix, classification report, kurva training, metrics

In [ ]:
model.save(MODELS / "waste_classifier.keras")
(MODELS / "class_names.json").write_text(
    json.dumps(class_labels, ensure_ascii=False, indent=2), encoding="utf-8")

report_dict = classification_report(
    y_true, y_pred, target_names=class_labels, output_dict=True)
metrics = {
    "test_accuracy": test_acc,
    "classes": class_labels,
    "images_per_split": {k[0] + "/" + k[1]: v for k, v in counts.items()},
    "total_images": sum(counts.values()),
    "model": "MobileNetV2 (transfer learning + fine-tuning)",
    "input_size": list(IMG_SIZE),
    "tensorflow_version": tf.__version__,
    "trained_at": datetime.now().isoformat(timespec="seconds"),
    "per_class": {c: {k: round(report_dict[c][k], 4) for k in ("precision", "recall", "f1-score")}
                  for c in class_labels},
}
(RESULTS / "metrics.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")

shutil.make_archive("/content/ml_artifacts", "zip", "/content/artifacts")
print("Selesai. Versi TensorFlow untuk requirements.txt:", tf.__version__)

In [ ]:
# Uji satu gambar dengan langkah praproses yang SAMA seperti API
from PIL import Image, ImageOps

def predict_image(path):
    img = ImageOps.exif_transpose(Image.open(path)).convert("RGB").resize(IMG_SIZE, Image.BILINEAR)
    arr = np.expand_dims(np.asarray(img, dtype="float32"), 0)   # nilai 0..255
    p = model(arr, training=False).numpy()[0]
    i = int(p.argmax())
    return class_labels[i], float(p[i])

for _, (dst, label) in CLASS_MAP.items():
    sample = sorted((OUT / "test" / dst).glob("*"))[0]
    print(f"{label:<15} -> {predict_image(sample)}")

In [ ]:
from google.colab import files
files.download("/content/ml_artifacts.zip")

## Langkah berikutnya

1. Ekstrak `ml_artifacts.zip` ke folder `ml/` di repo (menghasilkan `ml/models/` dan `ml/results/`).
2. Tulis versi TensorFlow yang tercetak di atas ke `ml/requirements.txt`.
3. Jalankan API: `uvicorn app.main:app --host 0.0.0.0 --port 8000` dari folder `ml/`.
4. Isi `EXPO_PUBLIC_ML_API_URL` di `mobile/.env` dengan alamat API.